# Chapter 15: Canard explosions — Brian2

Representative simulations adapted from [`python/chapter15.ipynb`](../python/chapter15.ipynb) and its [chapter notes](../python/chapter15.md).

Coverage: the FHN single-orbit CANARD_2 example with a=5, tau_n=60 and I=-4.256889, initialized at (0,0). A shorter 2000-unit integration displays its final 400 units, followed by a step-halving comparison. The reference integrates 10000 units and also performs fine amplitude/continuation scans; those and the HH/MMO examples remain in Python. The narrow canard window makes discretization checks essential.

$\dot v=v-v^3/3-n+I$, $\dot n=(5v-n)/60$. A canard tracks a repelling part of the cubic slow manifold before leaving it.

Run `uv sync --group dev` in the repository root and select that environment's Python kernel. These examples require the source checkout containing `mnd/brian`; an older PyPI installation does not contain these new helpers. Run all cells in order. Brian2 performs every trajectory integration. Numerical states use the book's units; helper clocks map a source time unit to one Brian millisecond. FHN and normal-form time is abstract model time.

In [ ]:
from pathlib import Path
import sys

# Use this source checkout so newly added helpers are available.
root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "mnd").is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))
import brian2 as b2
import numpy as np
import matplotlib.pyplot as plt
b2.prefs.codegen.target = "numpy"

In [ ]:
from mnd.brian.dynamics import simulate_fhn
coarse=simulate_fhn(a=5,tau_n=60,current=-4.256889,initial=(0,0),duration=2000,dt=.04)
fine=simulate_fhn(a=5,tau_n=60,current=-4.256889,initial=(0,0),duration=2000,dt=.02)
tail=fine.t>=1600
fig,ax=plt.subplots(1,2,figsize=(10,4))
vgrid=np.linspace(-2.5,2.5,400)
ax[0].plot(vgrid,vgrid-vgrid**3/3-4.256889,"k--",label="v nullcline")
ax[0].plot(fine.v[0,tail],fine.n[0,tail],label="Brian2 orbit")
ax[0].set(xlabel="v",ylabel="n");ax[0].legend()
ax[1].plot(fine.t[tail],fine.v[0,tail]);ax[1].set(xlabel="model time",ylabel="v")
plt.tight_layout();plt.show()
coarse_amplitude=np.ptp(coarse.v[0,coarse.t>=1600])
fine_amplitude=np.ptp(fine.v[0,tail])
print(f"Tail amplitudes: dt=.04 {coarse_amplitude:.5f}; dt=.02 {fine_amplitude:.5f}")
assert abs(coarse_amplitude-fine_amplitude)<.02
assert fine_amplitude>1